In [1]:
import pandas as pd
import numpy as np

print("Feature engineering started.")

Feature engineering started.


In [2]:
assets = pd.read_csv(
    "../data/processed/assets_clean.csv"
)

maintenance = pd.read_csv(
    "../data/processed/maintenance_clean.csv"
)

failures = pd.read_csv(
    "../data/processed/failures_clean.csv"
)

trains = pd.read_csv(
    "../data/processed/trains_clean.csv"
)

freight = pd.read_csv(
    "../data/processed/freight_clean.csv"
)

In [3]:
assets["last_maintenance_date"] = pd.to_datetime(
    assets["last_maintenance_date"]
)

maintenance["created_date"] = pd.to_datetime(
    maintenance["created_date"]
)

maintenance["due_date"] = pd.to_datetime(
    maintenance["due_date"]
)

failures["failure_date"] = pd.to_datetime(
    failures["failure_date"]
)

trains["entry_time"] = pd.to_datetime(
    trains["entry_time"]
)

trains["exit_time"] = pd.to_datetime(
    trains["exit_time"]
)

freight["date"] = pd.to_datetime(
    freight["date"]
)

In [4]:
current_date = pd.Timestamp("2026-09-03")

assets["asset_age_years"] = (
    current_date.year -
    assets["installation_year"]
)

print(
    assets[
        ["asset_id", "installation_year", "asset_age_years"]
    ]
)

  asset_id  installation_year  asset_age_years
0   TRK001               2012               14
1   TRK002               2020                6
2   SIG001               2016               10
3   SIG002               2022                4
4   OHE001               2014               12
5   OHE002               2019                7


In [5]:
assets["days_since_maintenance"] = (
    current_date -
    assets["last_maintenance_date"]
).dt.days

print(
    assets[
        [
            "asset_id",
            "last_maintenance_date",
            "days_since_maintenance"
        ]
    ]
)

  asset_id last_maintenance_date  days_since_maintenance
0   TRK001            2026-05-12                     114
1   TRK002            2026-07-10                      55
2   SIG001            2026-06-15                      80
3   SIG002            2026-08-01                      33
4   OHE001            2026-05-20                     106
5   OHE002            2026-07-22                      43


In [6]:
failure_counts = (
    failures
    .groupby("asset_id")
    .size()
    .reset_index(
        name="historical_failure_count"
    )
)

assets = assets.merge(
    failure_counts,
    on="asset_id",
    how="left"
)

assets["historical_failure_count"] = (
    assets["historical_failure_count"]
    .fillna(0)
)

print(
    assets[
        [
            "asset_id",
            "failure_count",
            "historical_failure_count"
        ]
    ]
)

  asset_id  failure_count  historical_failure_count
0   TRK001              7                       3.0
1   TRK002              1                       0.0
2   SIG001              4                       2.0
3   SIG002              0                       0.0
4   OHE001              5                       2.0
5   OHE002              1                       0.0


In [7]:
assets["failure_frequency_per_year"] = (
    assets["historical_failure_count"] /
    assets["asset_age_years"].clip(lower=1)
)

print(
    assets[
        [
            "asset_id",
            "historical_failure_count",
            "asset_age_years",
            "failure_frequency_per_year"
        ]
    ]
)

  asset_id  historical_failure_count  asset_age_years  \
0   TRK001                       3.0               14   
1   TRK002                       0.0                6   
2   SIG001                       2.0               10   
3   SIG002                       0.0                4   
4   OHE001                       2.0               12   
5   OHE002                       0.0                7   

   failure_frequency_per_year  
0                    0.214286  
1                    0.000000  
2                    0.200000  
3                    0.000000  
4                    0.166667  
5                    0.000000  


In [8]:
assets["failure_frequency_per_year"] = (
    assets["historical_failure_count"] /
    assets["asset_age_years"].clip(lower=1)
)

print(
    assets[
        [
            "asset_id",
            "historical_failure_count",
            "asset_age_years",
            "failure_frequency_per_year"
        ]
    ]
)


  asset_id  historical_failure_count  asset_age_years  \
0   TRK001                       3.0               14   
1   TRK002                       0.0                6   
2   SIG001                       2.0               10   
3   SIG002                       0.0                4   
4   OHE001                       2.0               12   
5   OHE002                       0.0                7   

   failure_frequency_per_year  
0                    0.214286  
1                    0.000000  
2                    0.200000  
3                    0.000000  
4                    0.166667  
5                    0.000000  


In [9]:
avg_failure_severity = (
    failures
    .groupby("asset_id")["severity"]
    .mean()
    .reset_index(
        name="average_failure_severity"
    )
)

assets = assets.merge(
    avg_failure_severity,
    on="asset_id",
    how="left"
)

assets["average_failure_severity"] = (
    assets["average_failure_severity"]
    .fillna(0)
)

print(
    assets[
        [
            "asset_id",
            "average_failure_severity"
        ]
    ]
)

  asset_id  average_failure_severity
0   TRK001                       9.0
1   TRK002                       0.0
2   SIG001                       7.5
3   SIG002                       0.0
4   OHE001                       8.5
5   OHE002                       0.0


In [10]:
downtime = (
    failures
    .groupby("asset_id")["downtime_hours"]
    .sum()
    .reset_index(
        name="historical_downtime_hours"
    )
)

assets = assets.merge(
    downtime,
    on="asset_id",
    how="left"
)

assets["historical_downtime_hours"] = (
    assets["historical_downtime_hours"]
    .fillna(0)
)

print(
    assets[
        [
            "asset_id",
            "historical_downtime_hours"
        ]
    ]
)

  asset_id  historical_downtime_hours
0   TRK001                       24.0
1   TRK002                        0.0
2   SIG001                        5.0
3   SIG002                        0.0
4   OHE001                       12.0
5   OHE002                        0.0


In [11]:
assets["condition_risk"] = (
    100 - assets["condition_score"]
) / 100

print(
    assets[
        [
            "asset_id",
            "condition_score",
            "condition_risk"
        ]
    ]
)

  asset_id  condition_score  condition_risk
0   TRK001               62            0.38
1   TRK002               88            0.12
2   SIG001               71            0.29
3   SIG002               94            0.06
4   OHE001               68            0.32
5   OHE002               91            0.09


In [12]:
overdue = (
    maintenance
    .groupby("asset_id")["overdue_days"]
    .max()
    .reset_index(
        name="max_overdue_days"
    )
)

assets = assets.merge(
    overdue,
    on="asset_id",
    how="left"
)

assets["max_overdue_days"] = (
    assets["max_overdue_days"]
    .fillna(0)
)

print(
    assets[
        [
            "asset_id",
            "max_overdue_days"
        ]
    ]
)

  asset_id  max_overdue_days
0   TRK001                 1
1   TRK002                 0
2   SIG001                 0
3   SIG002                 0
4   OHE001                 0
5   OHE002                 0


In [13]:
max_severity = (
    maintenance
    .groupby("asset_id")["severity"]
    .max()
    .reset_index(
        name="max_pending_severity"
    )
)

assets = assets.merge(
    max_severity,
    on="asset_id",
    how="left"
)

assets["max_pending_severity"] = (
    assets["max_pending_severity"]
    .fillna(0)
)

In [14]:
max_criticality = (
    maintenance
    .groupby("asset_id")["criticality"]
    .max()
    .reset_index(
        name="max_pending_maintenance_criticality"
    )
)

assets = assets.merge(
    max_criticality,
    on="asset_id",
    how="left"
)

assets["max_pending_maintenance_criticality"] = (
    assets[
        "max_pending_maintenance_criticality"
    ].fillna(0)
)

In [15]:
pending_tasks = (
    maintenance[
        maintenance["status"] == "PENDING"
    ]
    .groupby("asset_id")
    .size()
    .reset_index(
        name="pending_task_count"
    )
)

assets = assets.merge(
    pending_tasks,
    on="asset_id",
    how="left"
)

assets["pending_task_count"] = (
    assets["pending_task_count"]
    .fillna(0)
)

In [16]:
print(
    assets[
        [
            "asset_id",
            "department",
            "asset_type",
            "asset_age_years",
            "days_since_maintenance",
            "condition_score",
            "condition_risk",
            "historical_failure_count",
            "failure_frequency_per_year",
            "average_failure_severity",
            "historical_downtime_hours",
            "max_overdue_days",
            "max_pending_severity",
            "max_pending_maintenance_criticality",
            "pending_task_count"
        ]
    ]
)

  asset_id   department asset_type  asset_age_years  days_since_maintenance  \
0   TRK001  ENGINEERING      TRACK               14                     114   
1   TRK002  ENGINEERING      TRACK                6                      55   
2   SIG001          S&T     SIGNAL               10                      80   
3   SIG002          S&T     SIGNAL                4                      33   
4   OHE001     TRACTION        OHE               12                     106   
5   OHE002     TRACTION        OHE                7                      43   

   condition_score  condition_risk  historical_failure_count  \
0               62            0.38                       3.0   
1               88            0.12                       0.0   
2               71            0.29                       2.0   
3               94            0.06                       0.0   
4               68            0.32                       2.0   
5               91            0.09                       0.0  

In [17]:
assets.to_csv(
    "../data/processed/asset_features.csv",
    index=False
)

print("✅ Asset feature dataset saved.")

✅ Asset feature dataset saved.
